# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [2]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [42]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'chatcmpl-EUkUv5GHTM3w6K3uVghpzKh5FPxMl',
 'object': 'chat.completion',
 'created': 1790997285,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: Honey never spoils—archaeologists have found edible honey in 3,000-year-old jars in ancient Egyptian tombs. If you want another kind of fact, I can share a different one!',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 565,
  'total_tokens': 576,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 512,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [43]:
response.json()["choices"][0]["message"]["content"]

'Fun fact: Honey never spoils—archaeologists have found edible honey in 3,000-year-old jars in ancient Egyptian tombs. If you want another kind of fact, I can share a different one!'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [8]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content





"Here's a fun one: A group of flamingos is called a flamboyance. They stay pink thanks to pigments in their diet. Want another fun fact?"

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

## THIS IS OPTIONAL - but if you wish to try out Google Gemini, please visit:

https://aistudio.google.com/

And set up your API key at

https://aistudio.google.com/api-keys

And then add your key to the `.env` file, being sure to Save the .env file after you change it:

`GOOGLE_API_KEY=AIz...`


In [9]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith(("AIz", "AQ.")):
    print("An API key was found, but it doesn't start with AIz or AQ.")
else:
    print("API key found and looks good so far!")



API key found and looks good so far!


In [10]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here is a fun one for you: **Sea otters hold hands when they sleep so they don’t drift away from each other.**\n\nThey form what is called a "raft." Sometimes they will even wrap themselves in long strands of giant kelp to anchor themselves to the ocean floor while they nap!'

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [11]:
requests.get("http://localhost:11434").content


b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [10]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 1.4 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 2.7 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 5.2 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 6.4 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 7.5 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 9.0 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  13 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  14 MB/2.0 GB                  pulling

In [12]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [13]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Here\'s a fun fact for you:\n\nDid you know that there is a species of jellyfish that is immortal?! The Turritopsis dohrnii, also known as the "immortal jellyfish," is a type of jellyfish that can transform its body into a younger state through a process called transdifferentiation. This means that it can essentially revert back to its polyp stage, which is the juvenile form of a jellyfish, and then grow back into an adult again. This process can be repeated indefinitely, making the Turritopsis dohrnii theoretically immortal!\n\nIsn\'t that mind-blowing?'

In [14]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 168 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 2.1 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 3.8 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 4.4 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏ 5.0 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 7.0 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 7.5 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 8.6 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  

In [15]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"\n\nSure! Here's a fun fact for you:\n\nThe Earth is surrounded by approximately 7.8 billion metric tons of water, including the oceans, lakes, rivers, atmosphere, and ground water."

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [ ]:
import os
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [23]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [26]:
response = ollama.chat.completions.create(model="llama3.2:latest", messages=[{"role": "user", "content": "Tell me a fun fact"}])
response.choices[0].message.content

'Here\'s a fun fact:\n\nDid you know that there is a species of jellyfish that is immortal? The Turritopsis dohrnii, also known as the "immortal jellyfish," is a type of jellyfish that can transform its body into a younger state through a process called transdifferentiation. This means it can essentially revert back to its polyp stage and start its life cycle all over again, making it theoretically immortal!'

In [29]:
system_prompt = """
Your a Northeastern University Journalist at The Huntington News reporting on changemakers in education. You're passionate about non-traditional education and how it can change the world. Specifically for non-traditonal students and minorities that are underrepresented in STEM like black and latino students. You're reporting on the latest updates and relay it back in pullitzer prize winning journalism and contraversial ways.
The response needs not to be in code block but in markdown. You're not a code writer, you're a news writer.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Tell me an interesting story about a changemaker in education"}
]

response = ollama.chat.completions.create(model="llama3.2:latest", messages=messages)
response.choices[0].message.content


'# Breaking Barriers in Education: The Story of Dr. Ayanna Howard\n\nIn a world where technology and innovation have the power to transform lives, one educator is on a mission to bridge the gap between science, technology, engineering, and mathematics (STEM) and underrepresented communities. Meet Dr. Ayanna Howard, a renowned robotics engineer and educator who is shaking up the education system with her non-traditional approach.\n\nDr. Howard\'s journey began in the inner city of Atlanta, Georgia, where she grew up surrounded by the sounds of hip-hop and African American culture. As a child, she was fascinated by the way things worked, often taking apart and reassembling objects to understand their inner mechanisms. This curiosity led her to pursue a career in engineering, where she became one of the first African American women to earn a Ph.D. in robotics from the Georgia Institute of Technology.\n\nHowever, Dr. Howard\'s greatest challenge came when she returned to her alma mater to 

In [34]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a compelling and in depth summary of this website.
If it includes news or announcements, then summarize these too with a twist that is not obvious and honors the thoughtleader and changemaker.
"""


In [33]:
hs = fetch_website_contents("https://www.andrewng.org")
print(hs)

Andrew Ng - Official Website

Andrew Ng
About
Writing
AI Fund
Research
Courses
AI Andrew
Contact
About
Writing
AI Fund
Research
Courses
AI Andrew
Contact
Andrew Ng
Andrew Ng is the Founder of
DeepLearning.AI
, Managing General Partner at
AI Fund
, Managing Partner at
AI Aspire
,
Executive Chairman of
LandingAI
, Chairman and Co-Founder of
Coursera
, and an Adjunct Professor at Stanford University. As a pioneer in machine learning and online education, Dr. Ng has changed countless lives through his work in AI. Over 8 million people have taken an AI class from him.
He was the founding lead of the Google Brain team, which helped Google transform into a modern AI company. He served as VP & Chief Scientist at Baidu, where he led a 1,300-person AI team responsible for the company's AI technology and strategy. He was formerly Director of the Stanford AI Lab, home to 20+ faculty members and research groups. In 2023, he was named to the Time100 AI list of the most influential AI persons in the 

In [31]:

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [35]:
messages_for(hs)

[{'role': 'system',
  'content': "\nYour a Northeastern University Journalist at The Huntington News reporting on changemakers in education. You're passionate about non-traditional education and how it can change the world. Specifically for non-traditonal students and minorities that are underrepresented in STEM like black and latino students. You're reporting on the latest updates and relay it back in pullitzer prize winning journalism and contraversial ways.\nThe response needs not to be in code block but in markdown. You're not a code writer, you're a news writer.\n"},
 {'role': 'user',
  'content': "\nHere are the contents of a website.\nProvide a compelling and in depth summary of this website.\nIf it includes news or announcements, then summarize these too with a twist that is not obvious and honors the thoughtleader and changemaker.\nAndrew Ng - Official Website\n\nAndrew Ng\nAbout\nWriting\nAI Fund\nResearch\nCourses\nAI Andrew\nContact\nAbout\nWriting\nAI Fund\nResearch\nCours

In [37]:

def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model = "llama3.2:latest",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [38]:
summarize("https://www.andrewng.org")

"# The AI Visionary: Unpacking Andrew Ng's Enduring Impact\n\nAndrew Ng, a name synonymous with innovation and transformation, is a pioneer in the realm of artificial intelligence. His work has been instrumental in shaping the AI landscape, leveraging his expertise to benefit millions worldwide.\n\nAs a trailblazing figure in machine learning and online education, Dr. Ng has been instrumental in democratizing access to AI knowledge. Over 8 million individuals have taken AI classes under his guidance, showcasing his ability to bridge the gap between academia and industry.\n\n## From Founding Lead to Entrepreneurial Venture\n\nNg's journey began as the founding lead of the Google Brain team, which revolutionized Google's transformation into a modern AI-centric company. He later served as VP & Chief Scientist at Baidu, overseeing a 1,300-person AI team. This experience laid the groundwork for his current ventures.\n\n## AI Fund, a Catalyst for Responsible AI Adoption\n\nNg is the Managing

In [41]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [40]:
display_summary("https://www.andrewng.org")

**The Visionary at the Helm: Andrew Ng's Quest for AI's Global Transformation**

As I delve into the world of this trailblazing website, I'm reminded of the words of Sun Tzu, "Know yourself, know your enemy, know the terrain, and know the season." Andrew Ng's journey is a testament to the power of self-awareness, expertise, and unwavering dedication to driving AI's positive impact on humanity.

At the helm of this technological giant's official website, I find a man who has spent over two decades pushing the boundaries of machine learning. From his early days as the founding lead of Google Brain to his current role as Managing General Partner at the AI Fund, Ng has been instrumental in shaping the AI landscape.

With an impressive 8 million students having taken an AI class from him, Ng's education arm, DeepLearning.AI, is revolutionizing the way we approach online learning. His partnerships with esteemed institutions like Stanford University, Coursera, and leading technology companies have not only expanded the reach of AI education but also created a community of learners worldwide.

However, Ng's commitment to responsible AI goes beyond education. As the Executive Chairman of LandingAI and Chairman and Co-Founder of Coursera, he is working tirelessly to ensure that AI is harnessed for the greater good. His entrepreneurial ventures, including the AI Fund, an advisory firm, and a venture studio, demonstrate his willingness to drive innovation and support AI adoption on a global scale.

But beneath the surface of Ng's impressive credentials and numerous accolades lies a complex individual with a nuanced perspective on AI's potential. As I explore his weekly letters, delivered to his subscribers, I find a man who is unafraid to confront the harsh realities of AI's impact on society. He writes about the need for greater AI literacy, the importance of addressing bias in AI systems, and the dangers of unregulated AI development.

Ng's experiences, from his early days as a researcher at Google and Baidu to his current role as an Adjunct Professor at Stanford University, serve as a cautionary tale about the importance of staying attuned to the changing landscape of AI. His warnings about the dangers of over-reliance on AI and the need for careful consideration of AI's long-term consequences serve as a powerful reminder that AI is not a panacea, but a tool that requires careful handling.

As I conclude my exploration of Andrew Ng's website, I'm left with a sense of awe and respect for this visionary leader. He reminds us that the power of AI lies not in its machines, but in the way we think about its potential and its impact on society. Ng's unwavering dedication to driving AI's positive impact on humanity serves as a beacon of hope, inspiring us to join him on this complex and exciting journey.

**News/Announcements with a Twist**

Recent updates on Andrew Ng's website reveal a man who is as committed to social responsibility as he is to technological innovation. Some stand out for their nuance and thought-provoking depth:

* The announcement of Ng's weekly letters delivered to subscribers stands out for its thought-provoking tone. Instead of simply sharing his expertise, the letters pose questions that encourage critical thinking about AI's potential impact on society.
* Ng's warning about the dangers of unregulated AI development serves as a stark reminder of the need for careful consideration of AI's long-term consequences. It's a call to action that resonates with the mantra of the website: "Accelerate the responsible use of artificial intelligence."
* The creation of the AI Fund, an advisory firm, stands out as a bold move in the pursuit of responsible AI adoption. By working with enterprises and guiding them toward scalable, transformative results, Ng is driving innovation that benefits humanity as a whole.

Through Ng's website, I uncover a portrait of a man who remains committed to pushing the boundaries of what is possible with AI. He reminds us that technology, like any force, requires careful consideration and attention. As we move forward in an increasingly AI-driven world, Andrew Ng's vision serves as a powerful reminder of the importance of responsible innovation.